In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [33]:
# --- 1. THE DATA PREPARATION ---
text = "hello world"
# Get all unique characters in our dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Create lookup tables to go from character -> integer, and integer -> character
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

In [34]:
# --- 2. THE MODEL ARCHITECTURE ---
class CharLM(nn.Module):
    def __init__(self, vocab_size, context_length, embedding_dim):
        super().__init__()
        # nn.Embedding is a lookup table. 
        # It maps our character integers into trainable vectors of size 'embedding_dim'
        self.emb = nn.Embedding(vocab_size, embedding_dim)
        # We will flatten the embeddings into a single 1D array.
        # So the input size to our first linear layer is context_length * embedding_dim
        self.linear1 = nn.Linear(context_length * embedding_dim, 32)
        # The output size MUST equal vocab_size. We need a score for every possible character.
        self.linear2 = nn.Linear(32, vocab_size)
    def forward(self, x):
        # x is our input sequence of integers. Shape: (batch_size, context_length)
        # 1. Translate integers to embeddings. 
        x = self.emb(x) # New Shape: (batch_size, context_length, embedding_dim)
        # 2. Flatten the tensor. 
        # .view() physically reshapes the tensor. x.shape[0] keeps the batch_size identical, 
        # and -1 tells PyTorch "figure out the rest of the dimensions and smash them into one line".
        x = x.view(x.shape[0], -1) 
        # 3. Pass through our linear layers
        x = F.relu(self.linear1(x))
        x = self.linear2(x) # Output Shape: (batch_size, vocab_size)
        return x

In [35]:
# Instantiate the model. Let's say we look at 3 previous characters, and use an embedding size of 10.
model = CharLM(vocab_size=vocab_size, context_length=3, embedding_dim=10)

In [36]:
# Dummy input: let's pretend we pass in the sequence "hel", which translates to some integers [h_int, e_int, l_int]
# Shape is (1 batch, 3 characters)
dummy_input = torch.tensor([[stoi['h'], stoi['e'], stoi['l']]])

In [37]:
# Get prediction
logits = model(dummy_input)
print(logits.shape) # Output will be [1, 8] because "hello world" has 8 unique characters.

torch.Size([1, 8])


In [38]:
print(logits)

tensor([[-0.0440, -0.0822,  0.0110,  0.2060, -0.0976,  0.2340,  0.3573, -0.0514]],
       grad_fn=<AddmmBackward0>)


# Exercise

In [41]:
text = "pytorch"

chars = sorted(set(list(text)))
stoi = {ch: i for i,ch in enumerate(chars)}
itos = {i: ch for i,ch in enumerate(chars)}

In [42]:
print(list(enumerate(chars)))

[(0, 'c'), (1, 'h'), (2, 'o'), (3, 'p'), (4, 'r'), (5, 't'), (6, 'y')]


In [43]:
x = torch.tensor([[3,6],[6,5],[5,2],[2,4],[4,0]])
y = torch.tensor([5,2,4,0,1])

In [44]:
class CharLM(nn.Module):
    def __init__(self, vocab_size, emb_dim=4, context_len=2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.linear1 = nn.Linear(context_len * emb_dim, 12)
        self.linear2 = nn.Linear(12, vocab_size)
    def forward(self, x):
        x = self.emb(x)
        x = x.view(x.shape[0], -1)
        x = self.linear1(x)
        x = F.relu(x)
        x = self.linear2(x)
        return x

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [49]:
model = CharLM(vocab_size=vocab_size, context_len=2, emb_dim=4)
criterion = nn.CrossEntropyLoss() 
optimizer = optim.SGD(model.parameters(), lr=0.1)
model.to(device)

CharLM(
  (emb): Embedding(8, 4)
  (linear1): Linear(in_features=8, out_features=12, bias=True)
  (linear2): Linear(in_features=12, out_features=8, bias=True)
)

In [50]:
epochs = 100
for epoch in range(epochs):
    x = x.to(device)
    y = y.to(device)
    optimizer.zero_grad()
    output = model(x)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/100, Loss: {loss.item():.4f}")

Epoch 20/100, Loss: 0.9637
Epoch 40/100, Loss: 0.3528
Epoch 60/100, Loss: 0.1311
Epoch 80/100, Loss: 0.0643
Epoch 100/100, Loss: 0.0387


In [51]:
# 1. Set the model to evaluation mode. 
# (For this tiny model it does nothing, but for real models with Dropout or BatchNorm, skipping this will ruin your predictions).
model.eval()

# Let's test it with a specific context. 
# The context must be exactly 2 characters because we hardcoded context_len=2.
seed_context = "yt"

# 2. Convert the text prompt into a tensor of integers on the correct device
# We wrap it in an extra bracket to create a batch dimension of 1 -> Shape: [1, 2]
context_ints = [stoi[ch] for ch in seed_context]
inference_input = torch.tensor([context_ints]).to(device)

# 3. Turn off the Autograd engine
with torch.no_grad():
    # Pass the input through the model
    prediction_logits = model(inference_input) # Shape: [1, 7]
    
# 4. Find the index with the highest probability
# dim=1 means "look across the 7 vocabulary columns, not down the batches"
predicted_index = torch.argmax(prediction_logits, dim=1).item()

# 5. Translate the integer back to a character
predicted_char = itos[predicted_index]

print(f"Given the context '{seed_context}', the model predicts the next character is: '{predicted_char}'")

Given the context 'yt', the model predicts the next character is: 'o'
